# DepFuzz — inputs that survive a join

A join makes two columns of two tables dependent. Mutate one freely and the rows stop
matching, the query returns nothing, and the campaign learns nothing. Drawing joined
columns from a shared pool fixes exactly that.

In [ ]:
import os, sys, glob

ROOT = os.environ.get("BIGASTERISK_HOME") or os.path.abspath("..")

# Jars: a source checkout has them under modules/*/target, the Docker image under jars/.
JARS = sorted(glob.glob(f"{ROOT}/modules/*/target/scala-2.13/bigasterisk-*.jar")) \
    or sorted(glob.glob(f"{ROOT}/jars/bigasterisk-*.jar"))
if not JARS:
    raise SystemExit("No BigAsterisk jars found. Run: bin/sbt package")

FASTUTIL_JAR = os.environ.get("FASTUTIL_JAR") or next(iter(sorted(
    glob.glob(f"{ROOT}/jars/fastutil*.jar")
    + glob.glob(os.path.expanduser("~/Library/Caches/Coursier/**/fastutil-8.5.15.jar"), recursive=True)
    + glob.glob(os.path.expanduser("~/.cache/coursier/**/fastutil-8.5.15.jar"), recursive=True)
)), None)
if not FASTUTIL_JAR:
    raise SystemExit("fastutil jar not found. Run: bin/sbt package")

SPARK_JARS = ",".join(JARS + [FASTUTIL_JAR])
DATA = f"{ROOT}/examples/data"
sys.path.insert(0, f"{ROOT}/python")

## The data

Twelve orders across three customers. One of them, `o8`, is an outlier at
`99999` — every notebook here uses it as the thing to find.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import bigasterisk

spark = (bigasterisk.configure(SparkSession.builder)
    .master("local[2]")
    .appName("depfuzz-notebook")
    .config("spark.jars", SPARK_JARS)
    .config("spark.sql.adaptive.skewJoin.enabled", "false")
    .config("spark.ui.enabled", "false")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

orders = spark.read.schema("oid STRING, cid STRING, amount INT").csv(f"{DATA}/orders.txt")
customers = spark.read.schema("cid STRING, name STRING").csv(f"{DATA}/customers.txt")
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

orders.show()

## The same campaign, two strategies

In [ ]:
QUERY = ("SELECT c.name, SUM(o.amount) AS total "
         "FROM orders o JOIN customers c ON o.cid = c.cid "
         "WHERE o.amount > 100 GROUP BY c.name")
seeds = {"orders": orders, "customers": customers}
fuzzer = bigasterisk.fuzz(spark)

unaware = fuzzer.fuzz(QUERY, seeds, iterations=20, strategy="random", seed=7)
aware = fuzzer.fuzz(QUERY, seeds, iterations=20, strategy="co-dependent", seed=7)

print("random         empty results:", unaware.empty_results, "of", unaware.iterations)
print("co-dependent   empty results:", aware.empty_results, "of", aware.iterations)

## Check

A randomly generated join key essentially never matches.

In [ ]:
assert unaware.empty_results > aware.empty_results
assert aware.empty_results < aware.iterations
print("OK")